# Premier League Analytics — 03: Player Analysis

Radar charts, scatter + regression, and player similarity scoring using `src/similarity.py`.  

> **Note on player data**: football-data.co.uk provides match-level team data, not player-level stats. This notebook synthesises per-90 player metrics from team-level data where possible, or uses a bundled sample dataset. For full player-level analytics, run `data/fetch_data.py` with the optional `soccerdata` path.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.')))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

from src.similarity import find_similar_players
from src.charts import radar_chart

warnings.filterwarnings('ignore')

FIG = Path('outputs/figures')
FIG.mkdir(parents=True, exist_ok=True)

## 1. Load or Generate Player Stats

We use a representative sample of top Premier League forwards/midfielders (2023-24 season) with publicly reported per-90 stats.

In [ ]:
# Representative per-90 stats for a selection of 2023-24 Premier League players
# Source: widely reported FBref/Opta figures (approximate)
player_data = [
    # player, goals_per90, assists_per90, shots_per90, key_passes_per90, dribbles_per90, tackles_per90, interceptions_per90, aerials_won_per90
    ('Erling Haaland',     0.85, 0.12, 4.5, 0.8, 0.3, 0.3, 0.1, 1.8),
    ('Mohamed Salah',      0.52, 0.31, 3.1, 2.1, 1.8, 0.5, 0.3, 0.3),
    ('Bukayo Saka',        0.41, 0.38, 2.8, 2.4, 2.1, 1.0, 0.5, 0.2),
    ('Son Heung-min',      0.48, 0.22, 3.0, 1.7, 1.5, 0.6, 0.3, 0.2),
    ('Phil Foden',         0.45, 0.28, 2.7, 2.2, 1.6, 0.8, 0.4, 0.2),
    ('Marcus Rashford',    0.38, 0.20, 2.9, 1.4, 1.7, 0.5, 0.3, 0.3),
    ('Ollie Watkins',      0.55, 0.30, 3.2, 1.0, 0.9, 0.4, 0.2, 1.5),
    ('Cole Palmer',        0.52, 0.40, 2.6, 2.8, 1.4, 0.7, 0.4, 0.2),
    ('Dominic Solanke',    0.32, 0.15, 2.5, 0.9, 0.8, 0.4, 0.2, 1.6),
    ('Jarrod Bowen',       0.35, 0.28, 2.4, 1.6, 1.9, 0.8, 0.5, 0.4),
    ('Rodri',              0.08, 0.14, 0.9, 1.8, 0.6, 3.5, 2.1, 0.8),
    ('Declan Rice',        0.12, 0.18, 1.1, 2.0, 0.9, 3.0, 1.8, 0.5),
    ('Bruno Fernandes',    0.28, 0.34, 2.2, 3.1, 1.0, 1.2, 0.6, 0.3),
    ('Kevin De Bruyne',    0.18, 0.52, 2.0, 3.5, 1.1, 0.9, 0.5, 0.2),
    ('Martin Odegaard',    0.22, 0.32, 2.1, 3.2, 1.3, 1.5, 0.8, 0.2),
]

cols = ['player', 'goals_per90', 'assists_per90', 'shots_per90', 'key_passes_per90',
        'dribbles_per90', 'tackles_per90', 'interceptions_per90', 'aerials_won_per90']
players = pd.DataFrame(player_data, columns=cols)
print(players)

## 2. Radar Chart

In [ ]:
features = ['goals_per90', 'assists_per90', 'shots_per90', 'key_passes_per90',
            'dribbles_per90', 'tackles_per90']

fig, ax = plt.subplots(subplot_kw={'polar': True}, figsize=(7, 7))
radar_chart(
    players,
    players=['Erling Haaland', 'Cole Palmer', 'Rodri'],
    features=features,
    ax=ax,
    title='Player Comparison Radar (per 90 mins, 2023-24)'
)
fig.tight_layout()
fig.savefig(FIG / 'player_radar.png', dpi=150)
plt.show()
print(f'Saved: {FIG / "player_radar.png"}')

**Interpretation**: Haaland dominates the goalscoring and shooting axes; Rodri dominates defensive metrics; Cole Palmer strikes a balance across creativity and output metrics.

## 3. Scatter + Regression: Goals vs Shots per 90

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(players['shots_per90'], players['goals_per90'], s=80, color='teal', zorder=3)

for _, row in players.iterrows():
    ax.annotate(row['player'].split()[-1],
                (row['shots_per90'], row['goals_per90']),
                textcoords='offset points', xytext=(5, 3), fontsize=8)

m, b = np.polyfit(players['shots_per90'], players['goals_per90'], 1)
xs = np.linspace(players['shots_per90'].min(), players['shots_per90'].max(), 100)
ax.plot(xs, m * xs + b, 'r--', label='Linear fit')

r, p = __import__('scipy').stats.pearsonr(players['shots_per90'], players['goals_per90'])
ax.set_title(f'Goals vs Shots per 90  (r = {r:.2f}, p = {p:.3f})')
ax.set_xlabel('Shots per 90')
ax.set_ylabel('Goals per 90')
ax.legend()
fig.tight_layout()
fig.savefig(FIG / 'goals_vs_shots_scatter.png', dpi=150)
plt.show()

## 4. Player Similarity

In [ ]:
similar_to_haaland = find_similar_players(players, 'Erling Haaland', n=5, metric='euclidean')
print('Most similar players to Erling Haaland:')
print(similar_to_haaland.to_string(index=False))

In [ ]:
similar_to_rodri = find_similar_players(players, 'Rodri', n=5, metric='cosine')
print('Most similar players to Rodri (cosine):')
print(similar_to_rodri.to_string(index=False))

**Interpretation**: Haaland's similarity cluster should contain high-volume, goal-focused forwards (Watkins, Solanke). Rodri's cluster should return other deep-lying midfielders with high tackle/interception counts (Rice).